In [ ]:
"""
NCF v28 FINAL (Google Colab + CSV)
Pakeitimai:
- GAME_MIN=1000 (geriausias filtras)
- balance=True tik train, test naudoja natūralius duomenis
- evaluate() su batch inference (OOM prevencija)
- squeeze(dim=-1) vietoj squeeze()
Train: 80%, Test: 20%
"""

# Prijungiame Google Drive, kad galėtume pasiekti CSV failus ir išsaugoti modelius
from google.colab import drive
drive.mount('/content/drive/')

# Standartinės bibliotekos: operacinė sistema, matematika, duomenys
import os
import numpy as np          # Skaičiavimams su masyvais (vektoriai, matricos)
import pandas as pd         # Duomenų lentelėms (DataFrame)
import torch                # PyTorch - neuroninio tinklo karkasas
import torch.nn as nn       # Neuroninio tinklo sluoksnių biblioteka
from sklearn.preprocessing import MultiLabelBinarizer  # Žanrų one-hot kodavimui

# ─────────────────────────────────────────────────────────────────────────────
# KELIAI — nurodo kur ieškoti failų ir kur juos išsaugoti
# ─────────────────────────────────────────────────────────────────────────────
DRIVE_PATH = '/content/drive/MyDrive/baigiamasis'        # Aplankas su CSV failais Drive'e
MODEL_DIR  = '/content/drive/MyDrive/baigiamasis/models' # Aplankas modeliams išsaugoti
os.makedirs(MODEL_DIR, exist_ok=True)  # Sukuria aplanką jei jo nėra (exist_ok neleidžia klaidos)

# ─────────────────────────────────────────────────────────────────────────────
# HIPERPARAMETRAI — visi reguliuojami parametrai vienoje vietoje
# ─────────────────────────────────────────────────────────────────────────────
EMBED_DIM    = 64    # Embedding vektoriaus dydis - kiek skaičių reprezentuoja vartotoją/žaidimą
EPOCHS       = 15    # Kiek kartų modelis peržiūri visus treniravimo duomenis
BATCH_SIZE   = 4096  # Kiek eilučių paduodama į modelį vienu ypu treniravimo metu
LR           = 0.001 # Learning rate - žingsnio dydis optimizuojant (per didelis=nestabilu, per mažas=lėta)
WEIGHT_DECAY = 1e-4  # L2 regularizacija - baudžia už per didelius svorius (prevencija overfitting)
NEG_RATIO    = 2     # Kiek negatyvių pavyzdžių vienam pozityviam treniravimo metu (1:2)
USER_MIN     = 50    # Minimalus reviews skaičius vartotojui - filtruojame neaktyvius
GAME_MIN     = 1000  # Minimalus reviews skaičius žaidimui - paliekame tik populiarius
TEST_RATIO   = 0.2   # 20% duomenų testavimui, 80% treniravimui
RANDOM_SEED  = 42    # Fiksuotas atsitiktinumas - užtikrina atkartojamus rezultatus

# Nustato ar naudoti GPU (cuda) ar CPU - GPU ~10x greitesnis treniravimui
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {DEVICE}")

# ─────────────────────────────────────────────────────────────────────────────
# DUOMENŲ ĮKĖLIMAS IR TVARKYMAS
# ─────────────────────────────────────────────────────────────────────────────
def load_data():
    print("📂 Krauname duomenis iš CSV...")

    # Nuskaitome rekomendacijų duomenis - kiekviena eilutė = vienas vartotojo-žaidimo įrašas
    df_rec   = pd.read_csv(f'{DRIVE_PATH}/recommendations3.csv')
    # Nuskaitome žaidimų duomenis - kaina, reitingas ir kt.
    df_games = pd.read_csv(f'{DRIVE_PATH}/games3.csv')

    # Paliekame tik reikalingus stulpelius, copy() vengiame SettingWithCopyWarning
    df_rec = df_rec[['user_id', 'app_id', 'hours', 'is_recommended']].copy()
    print(f"   Raw: {len(df_rec)} eilučių")

    # Skaičiuojame kiek kiekvienas vartotojas turi reviews
    user_counts = df_rec['user_id'].value_counts()
    # Paliekame tik vartotojus su >=50 reviews - šalinami neaktyvūs vartotojai
    df_rec = df_rec[df_rec['user_id'].isin(user_counts[user_counts >= USER_MIN].index)]

    # Skaičiuojame kiek kiekvienas žaidimas turi reviews
    game_counts = df_rec['app_id'].value_counts()
    # Paliekame tik žaidimus su >=1000 reviews - mažiau žaidimų, bet kokybiškesni duomenys
    df_rec = df_rec[df_rec['app_id'].isin(game_counts[game_counts >= GAME_MIN].index)]
    print(f"   Po filtravimo: {len(df_rec)} eilučių")
    print(f"   Vartotojai: {df_rec['user_id'].nunique()}, Žaidimai: {df_rec['app_id'].nunique()}")

    # 1. Log transformacija žaidimo valandoms
    # log(1+hours) mažina didelių reikšmių dominavimą (pvz. 5000h → 8.5, 10h → 2.4)
    # clip(0,1000) apribojame maksimumą, nes outlieriai iškraipo modelį
    df_rec['hours_log'] = np.log1p(df_rec['hours'].clip(0, 1000))

    # 2. Paruošiame žaidimų features iš games3.csv
    df_games = df_games[['app_id', 'price_final', 'positive_ratio']].copy()
    df_games['price_final']    = df_games['price_final'].fillna(0)  # Trūkstama kaina = 0 (nemokamas)
    df_games['positive_ratio'] = df_games['positive_ratio'].fillna(df_games['positive_ratio'].median())  # Trūkstamas reitingas = mediana

    # 3. Normalizuojame kainą į [0,1] intervalą
    # Dalijame iš maksimalios kainos - modelis geriau dirba su mažais skaičiais
    price_max = df_games['price_final'].max()
    df_games['price_norm'] = df_games['price_final'] / (price_max + 1e-9)  # +1e-9 vengiame dalybos iš nulio

    # 4. Normalizuojame positive_ratio į [0,1] - buvo procentais (0-100), darome dešimtainį
    df_games['ratio_norm'] = df_games['positive_ratio'] / 100.0

    # 5. Žanrų one-hot kodavimas iš games_papildomas.csv
    df_pap = pd.read_csv(f'{DRIVE_PATH}/games_papildomas.csv')
    df_pap = df_pap[['AppID', 'Genres']].copy()
    # Pervadiname stulpelius kad sutaptų su kitais DataFrame
    df_pap = df_pap.rename(columns={'AppID': 'app_id', 'Genres': 'genres'})

    # AppID gali būti tekstas arba float - konvertuojame į skaičių, klaidas → NaN
    df_pap['app_id'] = pd.to_numeric(df_pap['app_id'], errors='coerce')
    # Šaliname eilutes kur app_id nėra teisingas skaičius (NaN arba begalybė)
    df_pap = df_pap[df_pap['app_id'].notna() & np.isfinite(df_pap['app_id'])]
    df_pap['app_id'] = df_pap['app_id'].astype(int)  # Konvertuojame į sveikąjį skaičių

    df_pap['genres'] = df_pap['genres'].fillna('')  # Trūkstami žanrai = tuščia eilutė

    # Padalijame žanrus į sąrašą: "Action,RPG" → ["action", "rpg"]
    df_pap['genres_list'] = df_pap['genres'].apply(
        lambda x: [g.strip().lower() for g in str(x).split(',') if g.strip()]
    )

    # MultiLabelBinarizer paverčia žanrų sąrašus į binary vektorius
    # pvz. ["action", "rpg"] → [1, 0, 0, 1, 0, ...]
    mlb = MultiLabelBinarizer()
    genres_encoded = mlb.fit_transform(df_pap['genres_list'])  # fit išmoksta visus žanrus, transform koduoja

    # Sukuriame stulpelių pavadinimus: genre_action, genre_rpg, ...
    genre_cols = [f'genre_{g}' for g in mlb.classes_]
    # Paverčiame numpy masyvą į DataFrame su tinkamais stulpeliais
    df_genres = pd.DataFrame(genres_encoded, columns=genre_cols, index=df_pap.index)
    # Sujungiame app_id su žanrų stulpeliais
    df_pap = pd.concat([df_pap[['app_id']], df_genres], axis=1)
    print(f"   Žanrai (one-hot): {len(genre_cols)} stulpeliai")

    # Sujungiame rekomendacijas su žaidimų duomenimis pagal app_id
    # how='left' paliekame visus rec įrašus, net jei žaidimas nerastas games lentelėje
    df_merged = df_rec.merge(df_games[['app_id', 'price_norm', 'ratio_norm']], on='app_id', how='left')
    # Prijungiame žanrų duomenis
    df_merged = df_merged.merge(df_pap, on='app_id', how='left')

    # Užpildome trūkstamas reikšmes po sujungimo - kai žaidimas nerastas papildomose lentelėse
    for col in ['price_norm', 'ratio_norm'] + genre_cols:
        df_merged[col] = df_merged[col].fillna(0)

    print(f"   Sujungtas DataFrame: {df_merged.shape}")
    return df_merged, genre_cols


# ─────────────────────────────────────────────────────────────────────────────
# TRAIN / TEST SPLIT — atskiria duomenis treniravimui ir testavimui
# ─────────────────────────────────────────────────────────────────────────────
def split_data(df):
    # Sumaišome duomenis atsitiktine tvarka - vengiame laiko ar kitokio šališkumo
    df = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    # Apskaičiuojame pjūvio tašką: 80% treniravimui
    split_idx = int(len(df) * (1 - TEST_RATIO))
    train_df = df.iloc[:split_idx].copy()  # Pirmieji 80% treniravimui
    test_df  = df.iloc[split_idx:].copy()  # Paskutiniai 20% testavimui
    print(f"\n   Train: {len(train_df)}, Test: {len(test_df)}")
    return train_df, test_df


# ─────────────────────────────────────────────────────────────────────────────
# NCF MODELIS — Neural Collaborative Filtering architektūra
# ─────────────────────────────────────────────────────────────────────────────
class NCF(nn.Module):
    def __init__(self, n_users, n_items, embed_dim, n_features):
        super().__init__()  # Inicializuojame tėvinę nn.Module klasę

        # Embedding lentelė vartotojams: kiekvienas vartotojas → embed_dim dydžio vektorius
        # Modelis pats išmoksta šiuos vektorius treniravimo metu
        self.user_emb = nn.Embedding(n_users, embed_dim)

        # Embedding lentelė žaidimams: kiekvienas žaidimas → embed_dim dydžio vektorius
        self.item_emb = nn.Embedding(n_items, embed_dim)

        # MLP įėjimo dydis = user embedding + item embedding + papildomi features
        input_dim = embed_dim * 2 + n_features

        # MLP (Multi-Layer Perceptron) - pilnai sujungti sluoksniai
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 128),  # Pirmasis sluoksnis: sumažiname dimensiją iki 128
            nn.ReLU(),                   # Aktyvacijos funkcija - įveda netiesiškumą (neigiamos → 0)
            nn.Dropout(0.2),             # Atsitiktinai "išjungia" 20% neuronų - prevencija overfitting
            nn.Linear(128, 64),          # Antrasis sluoksnis: 128 → 64
            nn.ReLU(),                   # Aktyvacijos funkcija
            nn.Linear(64, 1),            # Išėjimo sluoksnis: 64 → 1 (vienas skaičius)
            nn.Sigmoid()                 # Sigmoid paverčia į tikimybę [0,1]
        )

    def forward(self, user_idx, item_idx, features):
        u = self.user_emb(user_idx)   # Gauname vartotojo embedding pagal jo indeksą
        i = self.item_emb(item_idx)   # Gauname žaidimo embedding pagal jo indeksą
        # Sujungiame embeddings su papildomais features į vieną vektorių
        # torch.cat sujungia tensors išilgai dimensijos dim=1 (stulpeliai)
        return self.mlp(torch.cat([u, i, features], dim=1)).squeeze(dim=-1)
        # squeeze(dim=-1) pašalina paskutinę dimensiją: [batch,1] → [batch]
        # dim=-1 yra saugiau nei squeeze() be argumento - neleidžia klaidos kai batch_size=1


# ─────────────────────────────────────────────────────────────────────────────
# PARUOŠIMAS — paverčia DataFrame į PyTorch tensor'ius
# balance=True trainui (balansuota), balance=False testui (natūralu)
# ─────────────────────────────────────────────────────────────────────────────
def prepare_tensors(df, user_to_idx, item_to_idx, feature_cols, balance=True):
    # Filtruojame - paliekame tik tuos vartotojus ir žaidimus kurie yra indeksuose
    df = df[df['user_id'].isin(user_to_idx) & df['app_id'].isin(item_to_idx)].copy()

    # Konvertuojame vartotojo ID į eilės numerį (0,1,2,...) - modelis reikalauja skaičių nuo 0
    df['user_idx'] = df['user_id'].map(user_to_idx)
    # Konvertuojame žaidimo ID į eilės numerį
    df['item_idx'] = df['app_id'].map(item_to_idx)
    # Label: 1 jei rekomenduoja, 0 jei ne - tai ką modelis turi išmokti spėti
    df['label']    = (df['is_recommended'] == True).astype(float)

    if balance:
        # TRENIRAVIMUI: balansuojame duomenis nes natūraliai yra ~83% pozityvių
        # Nebalansuoti duomenys verčia modelį spėti viską kaip "teigiama"
        pos = df[df['label'] == 1]              # Visi pozityvūs įrašai
        neg_available = df[df['label'] == 0]    # Visi negatyvūs įrašai
        # Imame NEG_RATIO kartų daugiau negatyvių nei pozityvių (bet ne daugiau nei yra)
        neg_count = min(len(pos) * NEG_RATIO, len(neg_available))
        neg = neg_available.sample(neg_count, random_state=RANDOM_SEED)  # Atsitiktinė imtis
        # Sujungiame ir sumaišome pozityvius ir negatyvius
        df_final = pd.concat([pos, neg]).sample(frac=1, random_state=RANDOM_SEED)
        print(f"   Pos: {len(pos)}, Neg: {neg_count} (ratio 1:{NEG_RATIO}) [balansuota]")
    else:
        # TESTAVIMUI: natūralus pasiskirstymas - atspindi realų pasaulį
        # Jei balansintume testą, metrikos būtų iškreiptos ir neatspindėtų realių sąlygų
        df_final = df
        pos_count = int(df_final['label'].sum())   # Kiek teigiamų
        neg_count = len(df_final) - pos_count       # Kiek neigiamų
        print(f"   Pos: {pos_count}, Neg: {neg_count} [natūralus pasiskirstymas]")

    # Konvertuojame į PyTorch tensor'ius - tai modelio "maistas"
    u = torch.tensor(df_final['user_idx'].values, dtype=torch.long)    # Vartotojų indeksai (sveiki skaičiai)
    i = torch.tensor(df_final['item_idx'].values, dtype=torch.long)    # Žaidimų indeksai (sveiki skaičiai)
    f = torch.tensor(df_final[feature_cols].values, dtype=torch.float) # Features (slankiojo kablelio)
    l = torch.tensor(df_final['label'].values, dtype=torch.float)      # Etiketės 0 arba 1
    return u, i, f, l


# ─────────────────────────────────────────────────────────────────────────────
# TRENIRAVIMAS — optimizuoja modelio svorius
# ─────────────────────────────────────────────────────────────────────────────
def train(model, u, i, f, l):
    model = model.to(DEVICE)  # Perkeliame modelį į GPU jei yra

    # Adam optimizatorius - adaptuoja learning rate kiekvienam parametrui atskirai
    # weight_decay - L2 regularizacija, baudžia už per didelius svorius
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # BCELoss - Binary Cross Entropy Loss, tinkama binarinei klasifikacijai (0 arba 1)
    # Matuoja skirtumą tarp modelio spėjimų ir tikrų etikečių
    criterion = nn.BCELoss()

    # TensorDataset sujungia kelis tensor'ius į vieną duomenų rinkinį
    dataset = torch.utils.data.TensorDataset(u, i, f, l)
    # DataLoader automatiškai skaido duomenis į batch'us ir sumaišo kiekvienos epochos pradžioje
    loader  = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    print(f"\n   Treniruojame {EPOCHS} epochų (train dydis: {len(u)})...")
    model.train()  # Įjungia treniravimo režimą (aktyvuoja Dropout)

    for epoch in range(EPOCHS):
        total_loss = 0
        for u_b, i_b, f_b, l_b in loader:  # Iteruojame per batch'us
            # Perkeliame batch'ą į GPU kad skaičiavimai būtų greiti
            u_b, i_b, f_b, l_b = u_b.to(DEVICE), i_b.to(DEVICE), f_b.to(DEVICE), l_b.to(DEVICE)

            optimizer.zero_grad()           # Nuvalome ankstesnius gradientus (būtina prieš kiekvieną batch'ą)
            preds = model(u_b, i_b, f_b)   # Pirmyn praėjimas - gauname modelio spėjimus
            loss = criterion(preds, l_b)    # Apskaičiuojame nuostolį (kiek modelis klysta)
            loss.backward()                 # Atgal praėjimas - apskaičiuojame gradientus (kaip keisti svorius)
            optimizer.step()                # Atnaujiname svorius pagal gradientus
            total_loss += loss.item()       # Kaupiame nuostolį statistikai (item() ištraukia skaliarą)

        if (epoch + 1) % 5 == 0:  # Rodome progresą kas 5 epochas (ne kiekvieną - per daug spausdintų)
            print(f"   Epocha {epoch+1:>2}/{EPOCHS} — Loss: {total_loss/len(loader):.4f}")

    return model


# ─────────────────────────────────────────────────────────────────────────────
# TESTAVIMAS — įvertina modelį ant testavimo duomenų
# Naudojame batch inference - vengiame GPU atminties perpildymo (OOM)
# ─────────────────────────────────────────────────────────────────────────────
def evaluate(model, u, i, f, l):
    model.eval()  # Išjungia treniravimo režimą (išjungia Dropout - visi neuronai aktyvūs)

    # Sukuriame DataLoader testavimui - batch'ais vengiame OOM klaidos GPU atmintyje
    # shuffle=False - nekeičiame tvarkos, nes svarbu kad spėjimai atitiktų etiketes
    dataset = torch.utils.data.TensorDataset(u, i, f, l)
    loader  = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    all_preds = []  # Saugosime visus spėjimus čia
    with torch.no_grad():  # Išjungiame gradientų skaičiavimą - greičiau ir mažiau atminties
        for u_b, i_b, f_b, _ in loader:  # _ - etiketės nereikalingos spėjimui
            # Spėjame kiekvieną batch'ą atskirai - vengiame vienu ypu perkrauti GPU
            preds_batch = model(u_b.to(DEVICE), i_b.to(DEVICE), f_b.to(DEVICE))
            # Perkeliame į CPU ir pridedame prie sąrašo
            all_preds.extend(preds_batch.cpu().numpy())

    preds  = np.array(all_preds)  # Sujungiame visus spėjimus į vieną masyvą
    labels = l.numpy()            # Tikros etiketės kaip numpy masyvas

    # Konvertuojame tikimybes į klases: >=0.5 rekomenduoja, <0.5 nerekomenduoja
    predicted = (preds >= 0.5).astype(float)
    # Accuracy - dalis teisingų spėjimų iš visų spėjimų
    accuracy  = (predicted == labels).mean()

    # True Positive - teisingai nuspėta "rekomenduoja"
    tp = ((predicted == 1) & (labels == 1)).sum()
    # False Positive - neteisingai nuspėta "rekomenduoja" (iš tikrųjų nerekomenduoja)
    fp = ((predicted == 1) & (labels == 0)).sum()
    # False Negative - neteisingai nuspėta "nerekomenduoja" (iš tikrųjų rekomenduoja)
    fn = ((predicted == 0) & (labels == 1)).sum()

    # Precision - iš visų "rekomenduoja" spėjimų, kiek buvo teisingi
    precision = tp / (tp + fp + 1e-9)   # +1e-9 vengiame dalybos iš nulio
    # Recall - iš visų tikrų "rekomenduoja", kiek modelis sugebėjo rasti
    recall    = tp / (tp + fn + 1e-9)
    # F1 - harmoninis Precision ir Recall vidurkis - pagrindinis balansą atspindintis rodiklis
    f1        = 2 * precision * recall / (precision + recall + 1e-9)

    print(f"\n{'='*50}")
    print(f"TEST REZULTATAI — v28 FINAL (natūralus test set)")
    print(f"{'='*50}")
    print(f"   Test dydis:  {len(labels)}")
    print(f"   Pos/Neg:     {int(labels.sum())}/{int(len(labels)-labels.sum())}")
    print(f"   Accuracy:    {accuracy:.4f}")
    print(f"   Precision:   {precision:.4f}")
    print(f"   Recall:      {recall:.4f}")
    print(f"   F1:          {f1:.4f}")
    print(f"{'='*50}")

    return {"accuracy": round(float(accuracy), 4),
            "precision": round(float(precision), 4),
            "recall": round(float(recall), 4),
            "f1": round(float(f1), 4)}


# ─────────────────────────────────────────────────────────────────────────────
# PALEIDIMAS — pagrindinė programa
# ─────────────────────────────────────────────────────────────────────────────
# Įkeliame ir paruošiame visus duomenis
df, genre_cols = load_data()

# Features sąrašas - du skaitiniai + žanrų one-hot stulpeliai
feature_cols = ['price_norm', 'ratio_norm'] + genre_cols
n_features   = len(feature_cols)  # Bendras features skaičius (reikalingas modelio dydžiui)
print(f"\n   Iš viso features: {n_features}")

print("\n" + "="*50)
print("TRAIN / TEST SPLIT (80/20)")
print("="*50)
train_df, test_df = split_data(df)  # Padalijame į train ir test

# Sukuriame vartotojų ir žaidimų ID → indekso žodynus
# Modelis dirba su eilės numeriais (0,1,2,...), ne su originaliais ID
all_users   = sorted(df['user_id'].unique())  # Unikalūs vartotojai surikiuoti
all_items   = sorted(df['app_id'].unique())   # Unikalūs žaidimai surikiuoti
user_to_idx = {u: i for i, u in enumerate(all_users)}  # {user_id: 0, user_id2: 1, ...}
item_to_idx = {g: i for i, g in enumerate(all_items)}  # {app_id: 0, app_id2: 1, ...}

print("\n" + "="*50)
print("PARUOŠIAME DUOMENIS")
print("="*50)
# Train duomenys SU balansavimu - svarbu modelio mokymuisi
u_train, i_train, f_train, l_train = prepare_tensors(
    train_df, user_to_idx, item_to_idx, feature_cols, balance=True
)
# Test duomenys BEZ balansavimo - natūralus pasiskirstymas atspindi realybę
u_test, i_test, f_test, l_test = prepare_tensors(
    test_df, user_to_idx, item_to_idx, feature_cols, balance=False
)
print(f"   Train tensors: {len(u_train)}")
print(f"   Test tensors:  {len(u_test)}")

print("\n" + "="*50)
print("NCF MODELIS — v28 FINAL")
print(f"EMBED_DIM={EMBED_DIM}, EPOCHS={EPOCHS}, BATCH_SIZE={BATCH_SIZE}, LR={LR}")
print(f"Filtrai: users>={USER_MIN}, games>={GAME_MIN}, neg_ratio=1:{NEG_RATIO}")
print(f"Pataisymai: natūralus test set, batch evaluate, squeeze(dim=-1)")
print("="*50)

# Inicializuojame modelį su tinkamais dydžiais
model = NCF(n_users=len(all_users), n_items=len(all_items),
            embed_dim=EMBED_DIM, n_features=n_features)

# Treniruojame modelį ant train duomenų
model = train(model, u_train, i_train, f_train, l_train)
# Vertiname modelį ant testavimo duomenų (naudojame natūralų pasiskirstymą)
metrics = evaluate(model, u_test, i_test, f_test, l_test)

# Išsaugome modelį ir metaduomenis ateičiai (Flask integracija)
import pickle

# Išsaugome modelio svorius .pth formatu (PyTorch standartas)
model_path = os.path.join(MODEL_DIR, 'ncf_v28_final.pth')
torch.save(model.state_dict(), model_path)

# Metaduomenys reikalingi inference metu - kad galėtume konvertuoti ID į indeksus
metadata = {
    'user_to_idx': user_to_idx,   # Vartotojų ID → indeksų žodynas
    'item_to_idx': item_to_idx,   # Žaidimų ID → indeksų žodynas
    'feature_cols': feature_cols, # Features stulpelių sąrašas (tvarka svarbi!)
    'n_users': len(all_users),    # Bendras vartotojų skaičius (reikalingas Embedding dydžiui)
    'n_items': len(all_items),    # Bendras žaidimų skaičius
    'n_features': n_features,     # Bendras features skaičius
    'embed_dim': EMBED_DIM,       # Embedding dydis (reikalingas modelio rekonstrukcijai)
}
# Išsaugome pickle formatu - patogu saugoti Python objektus su žodynais
with open(os.path.join(MODEL_DIR, 'ncf_v28_final_metadata.pkl'), 'wb') as f:
    pickle.dump(metadata, f)

print(f"\n✅ Modelis išsaugotas: {model_path}")
print(f"📊 Metrikos: {metrics}")

In [ ]:
# Testavimas su konkrečiais žaidimais
df_games_raw = pd.read_csv(f'{DRIVE_PATH}/games3.csv')
id_to_title = dict(zip(df_games_raw['app_id'], df_games_raw['title']))

model.eval()
test_pairs = [
    (570,    "Dota 2"),
    (730,    "Counter-Strike: GO"),
    (489830, "Skyrim"),
    (578080, "PUBG"),
    (1091500,"Cyberpunk 2077"),
]

print("\n🎮 PAVYZDINĖS PROGNOZĖS (v28)")
print("="*50)
sample_user = list(user_to_idx.keys())[0]
u_idx = user_to_idx[sample_user]

for app_id, name in test_pairs:
    if app_id not in item_to_idx:
        print(f"   {name}: nėra duomenyse")
        continue
    i_idx = item_to_idx[app_id]
    game_row = df[df['app_id'] == app_id].iloc[0]
    feat = torch.tensor([game_row[col] for col in feature_cols], dtype=torch.float).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        score = model(
            torch.tensor([u_idx], dtype=torch.long).to(DEVICE),
            torch.tensor([i_idx], dtype=torch.long).to(DEVICE),
            feat
        ).item()
    result = '✅ rekomenduoja' if score >= 0.5 else '❌ nerekomenduoja'
    print(f"   {name:<35} -> {score:.4f} ({result})")